In [17]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

from bwLabel import labeling
from psColor import Gray2PseudoColor, CreateColorMap

%matplotlib inline

In [18]:
def show_image(img, title="Imagem", cmap=None):
    plt.figure(figsize=(8,6))
    if len(img.shape) == 3:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.imshow(img_rgb)
    else:
        plt.imshow(img, cmap=cmap if cmap else 'gray')
    plt.title(title)
    plt.axis('off')
    plt.show()        

In [19]:
def preprocess_and_binarize(image_path):
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Não foi possível carregar a imagem: {image_path}")

        blurred = cv2.GaussianBlur(img, (5, 5), 0)

        gray = cv2.cvtColor(blurred, cv2.COLOR_BGR2GRAY)

        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

        return img, binary

In [20]:
def apply_morphology(binary_img):
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))

    opened = cv2.morphologyEx(binary_img, cv2.MORPH_OPEN, kernel)

    cleaned = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel)

    return cleaned

In [21]:
def extract_lego_feature(binary_img):
    contours, hierarchy = cv2.findContours(binary_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    features_list = []

    for cnt in contours:
        area = cv2.contourArea(cnt)

        if area < 300:
            continue

        rect = cv2.minAreaRect(cnt)
        (center_x, center_y), (width, height), angle = rect

        w = max(width, height)
        h = min(width, height)

        aspect_ratio = w / h if h > 0 else 0

        features_list.append({
            'contour': cnt,
            'area': area,
            'width': w,
            'height': h,
            'aspect_ratio': aspect_ratio,
            'rect': rect
        })
    return features_list    

In [22]:
def classify_lego(feature):
    area = feature['area']
    aspect_ratio = feature['aspect_ratio']

    if 0.8 <= aspect_ratio < 1.4:
        return "2x2"
    elif 1.4 <= aspect_ratio < 2.4:
        return "2x4"
    elif 2.4 <= aspect_ratio < 3.4:
        return "2x6"
    elif 3.4 <= aspect_ratio < 4.6:
        return "2x8"
    else:
        return "Indef."                

In [23]:
def process_lego_image(image_path):
    orig_img, binary = preprocess_and_binarize(image_path)

    cleaned_binary = apply_morphology(binary)

    features = extract_lego_features(cleaned_binary)

    counts = {"2x2": 0, "2x4": 0, "2x6": 0, "2x8": 0, "Indef.": 0}

    out_img = orig_img.copy()

    for feat in features:
        label = classify_lego(feat)
        counts[label] += 1

        box = cv2.boxPoints(feat['rect'])
        box = np.int32(box)
        cv2.drawContours(out_img, [box], 0, (0, 255, 0), 2)

        M = cv2.moments(feat['contour'])
        if M["m00"] != 0:
            cX = int(M["m10"] / M["m00"])
            cY = int(M["m01"] / M["m00"])
        else:
            cX, cY = int(feat['rect'][0][0]), int(feat['rect'][0][1])
        cv2.putText(out_img, label, (cX - 20, cY),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

    print("--- CONTAGEM DE PEÇAS ---")
    for k, v in counts.items():
        if k!= "Indef.":
            print(f"Pecas {k}: {v}")
    if counts["Indef."] > 0:
        print(f"Outras / Indefinidas: {counts['Indef.']}")
    
    show_image(cleaned_binary, "Imagem Processada (Binária)")
    show_image(out_img, "Deteção e Classificação Final")

    num_labels, labeled_img = labeling(cleanded_binary)
    cmap = CreateColorMap(num_labels, firstColorBlack=1)
    pseudo_color_img = Gray2PseudoColor(labeled_img.astype(np.uint8), cmap)       
    show_image(pseudo_color_img, "Etiquetação em Falsa Cor (bwLabel + psColor)")                             

In [24]:
process_lego_image(image_path="imagens/lego01.jpg")

TypeError: cannot unpack non-iterable NoneType object